# Notebook 2.4e: Path-Level Marginal Analysis

**Purpose:** Quantify the marginal contribution of each origin-destination path to decongestion under different subsidy scenarios.

**Key Concepts:**
- **Existing paths**: Origin-destination pairs with observed student flow
- **Hypothetical paths**: Pairs generated as potential options but no observed flow
- **Marginal contribution**: Additional decongestion from increasing subsidy

**Policy Question:**
> "What is the marginal contribution of hypothetical paths to decongestion?"

**Related Documentation:** `references/documentation/dcm_informed_policy_simulation.md`

---
# 0. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "output"

print(f"Project directory: {PROJECT_DIR}")

Project directory: /workspace/project_paaral


---
# 1. Load Data

We need three datasets:
1. **Candidate Beneficiary Pool** — All origin-destination pairs with mu values
2. **Observed Student Flow** — To identify existing vs hypothetical paths
3. **Flow to Congested** — To link origins to congested public schools

In [2]:
# Candidate beneficiary pool (311K pairs with mu values)
cbp = pd.read_parquet(OUTPUT_DIR / "full_candidate_beneficiary_pool.parquet")
print(f"Candidate beneficiary pool: {cbp.shape[0]:,} pairs")

Candidate beneficiary pool: 311,074 pairs


In [3]:
# Observed student flow (existing paths)
student_flow = pd.read_parquet(OUTPUT_DIR / "grade_7_student_flow_table_sy2324.parquet")
print(f"Observed student flow: {student_flow.shape[0]:,} pairs")
print(f"Columns: {student_flow.columns.tolist()}")

Observed student flow: 288,328 pairs
Columns: ['school_id_origin', 'school_id_destination', 'count_non_beneficiary', 'count_esc_beneficiary']


In [5]:
# Flow to congested public schools
flow_to_congested = pd.read_parquet(OUTPUT_DIR / "analysis_payload" / "flow_to_congested.parquet")
print(f"Flow to congested: {flow_to_congested.shape[0]:,} rows")
print(f"Columns: {flow_to_congested.columns.tolist()}")

Flow to congested: 36,164 rows
Columns: ['school_id_origin', 'school_id_destination', 'count_non_beneficiary', 'count_esc_beneficiary', 'count_aisle_learner_jhs', 'seat_utilization_jhs', 'enrollment_jhs', 'seat_count']


---
# 2. Create `is_hypothetical` Flag

A path is **existing** if it appears in the observed student flow data (any flow, ESC or non-beneficiary).

A path is **hypothetical** if it was generated but has no observed flow.

In [6]:
# Ensure consistent types for merging
cbp['origin_school_id'] = cbp['origin_school_id'].astype(str)
cbp['destination_school_id'] = cbp['destination_school_id'].astype(str)

student_flow['school_id_origin'] = student_flow['school_id_origin'].astype(str)
student_flow['school_id_destination'] = student_flow['school_id_destination'].astype(str)

In [7]:
# Create set of existing (origin, destination) pairs
existing_pairs = set(
    zip(student_flow['school_id_origin'], student_flow['school_id_destination'])
)
print(f"Existing pairs in flow data: {len(existing_pairs):,}")

Existing pairs in flow data: 288,328


In [8]:
# Flag hypothetical paths
cbp['is_hypothetical'] = ~cbp.apply(
    lambda row: (row['origin_school_id'], row['destination_school_id']) in existing_pairs,
    axis=1
)

print(f"\nPath classification:")
print(cbp['is_hypothetical'].value_counts().rename({True: 'Hypothetical', False: 'Existing'}))


Path classification:
is_hypothetical
Hypothetical    234972
Existing         76102
Name: count, dtype: int64


---
# 3. Compute Proportional Weights for Congested Schools

An origin school may feed students into multiple congested public schools. We compute the **proportional weight** based on observed flow volume.

$$
\text{weight}_{\text{origin} \rightarrow \text{congested}_C} = \frac{\text{flow to } C}{\text{total flow to all congested schools}}
$$

In [9]:
# Ensure consistent types
flow_to_congested['school_id_origin'] = flow_to_congested['school_id_origin'].astype(str)
flow_to_congested['school_id_destination'] = flow_to_congested['school_id_destination'].astype(str)

# Check columns for flow count
print("Columns in flow_to_congested:")
print(flow_to_congested.columns.tolist())

Columns in flow_to_congested:
['school_id_origin', 'school_id_destination', 'count_non_beneficiary', 'count_esc_beneficiary', 'count_aisle_learner_jhs', 'seat_utilization_jhs', 'enrollment_jhs', 'seat_count']


In [10]:
# Inspect a few rows to understand the data
display(flow_to_congested.head())

,school_id_origin,school_id_destination,count_non_beneficiary,count_esc_beneficiary,count_aisle_learner_jhs,seat_utilization_jhs,enrollment_jhs,seat_count
0,101701,301194,1.0,NaN,4523.0,2.847631,6971.0,2448.0
1,101701,301215,10.0,NaN,923.0,1.419164,3125.0,2202.0
2,101701,307813,1.0,NaN,1178.0,1.583746,3196.0,2018.0
3,101782,301008,6.0,NaN,127.0,1.481061,391.0,264.0
4,101788,301008,4.0,NaN,127.0,1.481061,391.0,264.0


In [11]:
# Identify the flow count column (adjust if different)
# Common names: 'count', 'flow', 'count_non_beneficiary', etc.
flow_col = 'count_non_beneficiary'  # Adjust based on inspection above

# Compute total flow per origin to all congested schools
origin_total_flow = flow_to_congested.groupby('school_id_origin')[flow_col].sum().reset_index()
origin_total_flow.columns = ['school_id_origin', 'total_flow_to_congested']

# Merge back to get proportional weight
flow_weights = flow_to_congested.merge(origin_total_flow, on='school_id_origin')
flow_weights['proportional_weight'] = flow_weights[flow_col] / flow_weights['total_flow_to_congested']

# Keep relevant columns
flow_weights = flow_weights[['school_id_origin', 'school_id_destination', 'proportional_weight']]
flow_weights.columns = ['origin_school_id', 'congested_school_id', 'proportional_weight']

print(f"Flow weights computed: {flow_weights.shape[0]:,} origin-congested pairs")
display(flow_weights.head())

Flow weights computed: 36,164 origin-congested pairs


,origin_school_id,congested_school_id,proportional_weight
0,101701,301194,0.083333
1,101701,301215,0.833333
2,101701,307813,0.083333
3,101782,301008,1.000000
4,101788,301008,1.000000


---
# 4. Join Paths with Congested School Weights

Each path (origin → ESC) is linked to the congested school(s) the origin feeds into.

In [12]:
# Join CBP with flow weights (left join to keep all paths)
paths = cbp.merge(
    flow_weights,
    on='origin_school_id',
    how='left'
)

print(f"Paths after join: {paths.shape[0]:,} rows")
print(f"(Paths are exploded — one row per origin-ESC-congested triple)")

Paths after join: 5,259,439 rows
(Paths are exploded — one row per origin-ESC-congested triple)


In [15]:
display(paths.head())

,origin_school_id,destination_school_id,origin_sector,origin_region,destination_sector,tuition_fees_k,destination_tuition_fees,distance_meters,distance_km,net_cost,net_cost_k,esc_rating,log_distance,log_tuition,log_net_cost_k,destination_jhs_enrollment,esc_amount,esc_amount_k,candidate_pool,mu_baseline_0_subsidy,mu_with_current_subsidy,mu_with_minus_1k_net_cost,mu_with_minus_10k_net_cost,mu_with_minus_15k_net_cost,mu_with_minus_20k_net_cost,net_cost_baseline,net_cost_current,net_cost_minus_1k,net_cost_minus_10k,net_cost_minus_15k,net_cost_minus_20k,prob_dist_with_0_subsidy,prob_dist_with_current_subsidy,prob_dist_minus_1k_net_cost,prob_dist_minus_10k_net_cost,prob_dist_minus_15k_net_cost,prob_dist_minus_20k_net_cost,is_hypothetical,congested_school_id,proportional_weight
0,104408,401116,Public,Region III (Central Luzon),Private,18.525,18525.0,215159.4,215.1594,9525.0,9.525,0.0,5.376016,2.971696,2.353753,949.0,9000.0,9.0,71.0,0.577287,0.661182,0.675836,1.106204,1.106204,1.106204,18.525,9.525,8.525,0.01,0.01,0.01,"[0.5953454904899813, 0.27840087316786827, 0.09...","[0.5567431371081846, 0.29017464886274685, 0.10...","[0.5503636288482983, 0.2918379159817853, 0.108...","[0.40107461083502377, 0.30611764208467857, 0.1...","[0.40107461083502377, 0.30611764208467857, 0.1...","[0.40107461083502377, 0.30611764208467857, 0.1...",False,NaN,0.0
1,104412,401116,Public,Region III (Central Luzon),Private,18.525,18525.0,216821.4,216.8214,9525.0,9.525,0.0,5.383675,2.971696,2.353753,949.0,9000.0,9.0,83.0,0.575594,0.659243,0.673854,1.102960,1.102960,1.102960,18.525,9.525,8.525,0.01,0.01,0.01,"[0.5961627566386738, 0.2781203880291121, 0.091...","[0.5575950602087365, 0.28994639014401624, 0.10...","[0.5512204511567828, 0.2916192558338635, 0.108...","[0.4019737900273903, 0.3061825537413981, 0.163...","[0.4019737900273903, 0.3061825537413981, 0.163...","[0.4019737900273903, 0.3061825537413981, 0.163...",False,300693,0.5
2,104412,401116,Public,Region III (Central Luzon),Private,18.525,18525.0,216821.4,216.8214,9525.0,9.525,0.0,5.383675,2.971696,2.353753,949.0,9000.0,9.0,83.0,0.575594,0.659243,0.673854,1.102960,1.102960,1.102960,18.525,9.525,8.525,0.01,0.01,0.01,"[0.5961627566386738, 0.2781203880291121, 0.091...","[0.5575950602087365, 0.28994639014401624, 0.10...","[0.5512204511567828, 0.2916192558338635, 0.108...","[0.4019737900273903, 0.3061825537413981, 0.163...","[0.4019737900273903, 0.3061825537413981, 0.163...","[0.4019737900273903, 0.3061825537413981, 0.163...",False,320001,0.5
3,104457,401070,Public,Region III (Central Luzon),Private,13.350,13350.0,95035.2,95.0352,4350.0,4.350,0.0,4.564715,2.663750,1.677097,1794.0,9000.0,9.0,85.0,0.843078,1.047029,1.095700,1.509887,1.509887,1.509887,13.350,4.350,3.350,0.01,0.01,0.01,"[0.48431624529358064, 0.3041547620049902, 0.13...","[0.41793173862783606, 0.3070119872816575, 0.15...","[0.4039961651381024, 0.3063214156704603, 0.163...","[0.3080554238795605, 0.28830476938616373, 0.18...","[0.3080554238795605, 0.28830476938616373, 0.18...","[0.3080554238795605, 0.28830476938616373, 0.18...",False,300787,0.2
4,104457,401070,Public,Region III (Central Luzon),Private,13.350,13350.0,95035.2,95.0352,4350.0,4.350,0.0,4.564715,2.663750,1.677097,1794.0,9000.0,9.0,85.0,0.843078,1.047029,1.095700,1.509887,1.509887,1.509887,13.350,4.350,3.350,0.01,0.01,0.01,"[0.48431624529358064, 0.3041547620049902, 0.13...","[0.41793173862783606, 0.3070119872816575, 0.15...","[0.4039961651381024, 0.3063214156704603, 0.163...","[0.3080554238795605, 0.28830476938616373, 0.18...","[0.3080554238795605, 0.28830476938616373, 0.18...","[0.3080554238795605, 0.28830476938616373, 0.18...",False,300800,0.2


In [13]:
# Handle origins not feeding any congested school
no_congested = paths['congested_school_id'].isna()
print(f"Paths without congested linkage: {no_congested.sum():,}")

# Set weight to 0 for these (they don't contribute to decongestion)
paths.loc[no_congested, 'proportional_weight'] = 0

Paths without congested linkage: 205


---
# 5. Compute Decongestion Contributions

For each path, compute:
- `decongestion_X = mu_X × proportional_weight`
- `marginal_decong_X = (mu_X - mu_current) × proportional_weight`

In [14]:
# Define scenarios
SCENARIOS = {
    'baseline': 'mu_baseline_0_subsidy',
    'current': 'mu_with_current_subsidy',
    'minus_1k': 'mu_with_minus_1k_net_cost',
    'minus_10k': 'mu_with_minus_10k_net_cost',
    'minus_15k': 'mu_with_minus_15k_net_cost',
    'minus_20k': 'mu_with_minus_20k_net_cost',
}

print("Scenarios:")
for name, col in SCENARIOS.items():
    print(f"  {name}: {col}")

Scenarios:
  baseline: mu_baseline_0_subsidy
  current: mu_with_current_subsidy
  minus_1k: mu_with_minus_1k_net_cost
  minus_10k: mu_with_minus_10k_net_cost
  minus_15k: mu_with_minus_15k_net_cost
  minus_20k: mu_with_minus_20k_net_cost


In [16]:
# Compute decongestion contribution for each scenario
for name, mu_col in SCENARIOS.items():
    # Decongestion = mu × weight
    paths[f'decong_{name}'] = paths[mu_col] * paths['proportional_weight']

print("Decongestion columns created.")

Decongestion columns created.


In [17]:
# Compute marginal decongestion (vs current subsidy)
baseline_col = SCENARIOS['current']

for name, mu_col in SCENARIOS.items():
    if name == 'current':
        continue
    # Marginal = (mu_scenario - mu_current) × weight
    paths[f'marginal_{name}'] = (paths[mu_col] - paths[baseline_col]) * paths['proportional_weight']

print("Marginal decongestion columns created.")

Marginal decongestion columns created.


In [18]:
# Inspect sample results
result_cols = [
    'origin_school_id', 'destination_school_id', 'congested_school_id',
    'is_hypothetical', 'proportional_weight',
    'decong_current', 'decong_minus_10k', 'marginal_minus_10k'
]

display(paths[result_cols].head(10))

,origin_school_id,destination_school_id,congested_school_id,is_hypothetical,proportional_weight,decong_current,decong_minus_10k,marginal_minus_10k
0,104408,401116,NaN,False,0.0,0.000000,0.000000,0.000000
1,104412,401116,300693,False,0.5,0.329621,0.551480,0.221858
2,104412,401116,320001,False,0.5,0.329621,0.551480,0.221858
3,104457,401070,300787,False,0.2,0.209406,0.301977,0.092572
4,104457,401070,300800,False,0.2,0.209406,0.301977,0.092572
5,104457,401070,300828,False,0.2,0.209406,0.301977,0.092572
6,104457,401070,301047,False,0.2,0.209406,0.301977,0.092572
7,104457,401070,305382,False,0.2,0.209406,0.301977,0.092572
8,104459,401365,500751,False,1.0,0.886449,1.068162,0.181713
9,104460,402529,300765,False,0.2,0.116782,0.154532,0.037749


---
# 6. Summary Statistics

In [24]:
# Total decongestion by path type (existing vs hypothetical)
summary = paths.groupby('is_hypothetical').agg({
    'decong_current': 'sum',
    'decong_minus_1k': 'sum',
    'decong_minus_10k': 'sum',
    'decong_minus_15k': 'sum',
    'decong_minus_20k': 'sum',
}).round(1)

summary.index = summary.index.map({True: 'Hypothetical', False: 'Existing'})

print("=== Total Decongestion by Path Type ===")
display(summary.style.format("{:,.0f}"))

=== Total Decongestion by Path Type ===


,decong_current,decong_minus_1k,decong_minus_10k,decong_minus_15k,decong_minus_20k
is_hypothetical,,,,,
Existing,"57,741","58,916","73,400","81,808","88,845"
Hypothetical,"97,402","98,612","116,052","131,024","143,436"


In [25]:
# Marginal contribution by path type
marginal_summary = paths.groupby('is_hypothetical').agg({
    'marginal_minus_1k': 'sum',
    'marginal_minus_10k': 'sum',
    'marginal_minus_15k': 'sum',
    'marginal_minus_20k': 'sum',
}).round(1)

marginal_summary.index = marginal_summary.index.map({True: 'Hypothetical', False: 'Existing'})

print("=== Marginal Decongestion (vs Current Subsidy) ===")
display(marginal_summary.style.format("{:,.0f}"))

=== Marginal Decongestion (vs Current Subsidy) ===


,marginal_minus_1k,marginal_minus_10k,marginal_minus_15k,marginal_minus_20k
is_hypothetical,,,,
Existing,"1,175","15,659","24,067","31,104"
Hypothetical,"1,211","18,651","33,623","46,035"


In [21]:
# Percentage from hypothetical paths
total_decong = paths.groupby('is_hypothetical')['decong_minus_10k'].sum()
pct_hypothetical = total_decong[True] / total_decong.sum() * 100

print(f"\nHypothetical paths contribute {pct_hypothetical:.1f}% of total decongestion (at -10k scenario)")


Hypothetical paths contribute 61.3% of total decongestion (at -10k scenario)


---
# 7. Aggregation by Congested School

In [23]:
# Decongestion by congested school
by_congested = paths.groupby('congested_school_id').agg({
    'decong_current': 'sum',
    'decong_minus_1k': 'sum',
    'marginal_minus_1k': 'sum',
}).round(1)

by_congested = by_congested.sort_values('decong_minus_1k', ascending=False)

print("=== Top 15 Congested Schools by Decongestion (at -1k) ===")
display(by_congested.head(15))

=== Top 15 Congested Schools by Decongestion (at -1k) ===


,decong_current,decong_minus_1k,marginal_minus_1k
congested_school_id,,,
301190,2315.1,2332.8,17.6
301196,2195.1,2215.7,20.6
305330,2063.4,2086.8,23.5
301192,1961.8,1989.2,27.4
301186,1883.0,1920.7,37.7
301180,1765.0,1790.2,25.3
301171,1524.6,1531.9,7.3
301516,1151.8,1169.5,17.7
305413,1160.3,1168.3,8.1


In [27]:
# Breakdown by congested school AND path type
by_congested_type = paths.groupby(['congested_school_id', 'is_hypothetical']).agg({
    'decong_minus_1k': 'sum',
}).round(1).unstack(fill_value=0)

by_congested_type.columns = ['Existing', 'Hypothetical']
by_congested_type['Total'] = by_congested_type['Existing'] + by_congested_type['Hypothetical']
by_congested_type['Pct_Hypothetical'] = (by_congested_type['Hypothetical'] / by_congested_type['Total'] * 100).round(1)

by_congested_type = by_congested_type.sort_values('Total', ascending=False)

print("=== Decongestion by Congested School (minus 1k): Existing vs Hypothetical ===")
display(by_congested_type.head(15))

=== Decongestion by Congested School (minus 1k): Existing vs Hypothetical ===


,Existing,Hypothetical,Total,Pct_Hypothetical
congested_school_id,,,,
301190,684.2,1648.5,2332.7,70.7
301196,676.6,1539.1,2215.7,69.5
305330,213.1,1873.8,2086.9,89.8
301192,857.9,1131.3,1989.2,56.9
301186,221.7,1699.0,1920.7,88.5
301180,724.1,1066.1,1790.2,59.6
301171,460.0,1071.9,1531.9,70.0
301516,379.8,789.7,1169.5,67.5
305413,356.0,812.4,1168.4,69.5


---
# 8. Export Results

In [ ]:
# Create export directory
EXPORT_DIR = OUTPUT_DIR / "path_marginal_analysis"
EXPORT_DIR.mkdir(exist_ok=True)

# Export path-level results
paths.to_parquet(EXPORT_DIR / "path_level_results.parquet", index=False)
print(f"✓ Exported path_level_results.parquet ({paths.shape[0]:,} rows)")

# Export summary by congested school
by_congested_type.to_csv(EXPORT_DIR / "decongestion_by_congested_school.csv")
print(f"✓ Exported decongestion_by_congested_school.csv")

print(f"\nAll exports saved to: {EXPORT_DIR}")